In [1]:
import pandas as pd
import numpy as np
import xarray as xr

In [2]:
df = pd.read_csv("/file_path")

In [3]:
# this cell most closely resembles my xarray version of the code found in the tropocalc 
# util file. 

def compute_tropopause_pressure(p, T, lapseC=2.0, debug=False):
    g = 9.80665
    R = 287.05
    const = g / R
    pMin = 85 #hpa
    pMax = 450.0 # hpa
    dZ = 2000.0 # m

    nLev = p.size
    nLevm = nLev - 1
    lapse = np.full(nLevm, np.nan)
    pHalf = np.full(nLevm, np.nan)
    pTrop = 0.0
    found = False

    for i in range(nLevm):
        # Skip invalid or duplicate levels
        if (
            p[i] <= 0 or p[i + 1] <= 0 or
            T[i] <= 0 or T[i + 1] <= 0 or
            p[i] == p[i + 1] or T[i] == T[i + 1]
        ):
            continue

        logT = np.log(T[i] / T[i + 1])
        logP = np.log(p[i] / p[i + 1])

        if np.isinf(logT) or np.isinf(logP) or np.isnan(logT) or np.isnan(logP) or logP == 0:
            continue

        lapse[i] = const * logT / logP
        pHalf[i] = 0.5 * (p[i] + p[i + 1])

    for i in range(nLevm - 1):
        if np.isnan(lapse[i]) or np.isnan(p[i]):
            continue

        if lapse[i] < lapseC / 1000.0 and p[i] < pMax and not found:
            if np.isnan(pHalf[i]) or np.isnan(pHalf[i + 1]):
                continue

            P1 = np.log(pHalf[i])
            P2 = np.log(pHalf[i + 1])
            if lapse[i] != lapse[i + 1] and not np.isnan(lapse[i + 1]):
                weight = (lapseC / 1000.0 - lapse[i]) / (lapse[i + 1] - lapse[i])
                pTrop = np.exp(P1 + weight * (P2 - P1))
            else:
                pTrop = pHalf[i]

            if T[i] <= 0:
                continue

            p2km = pTrop * np.exp(-dZ * const / T[i])
            lapseSum = 0
            kount = 0
            for L in range(i, nLevm):
                if not np.isnan(pHalf[L]) and pHalf[L] > p2km and not np.isnan(lapse[L]):
                    lapseSum += lapse[L]
                    kount += 1
            lapseAvg = lapseSum / kount if kount > 0 else lapse[i]
            found = lapseAvg < lapseC / 1000.0
            if found:
                if pTrop < pMin and debug:
                    print(f"Warning: Tropopause pressure unusually low ({pTrop:.2f} hPa)")
                return pTrop  # in Pa

    if debug:
        print("No valid tropopause found. Check profile quality.")
        print("Pressure range:", np.nanmin(p), "to", np.nanmax(p))
        print("Temperature range:", np.nanmin(T), "to", np.nanmax(T))
        print("Lapse rate sample:", lapse[:10])

    return None  

p = df["pressure_hPa"] #.values * 100
T = df["temperature_C"].values + 273.15

ptrop = compute_tropopause_pressure(p, T, debug=True)
print(ptrop)
lat = df["latitude"].iloc[0]
lon = df["longitude"].iloc[0]

if ptrop and not np.isnan(ptrop):
    print(f"Tropopause pressure: {ptrop:.2f} hPa at lat={lat:.2f}, lon={lon:.2f}")
else:
    print(f"No tropopause found for profile at lat={lat:.2f}, lon={lon:.2f}")


29.11448270597697
Tropopause pressure: 29.11 hPa at lat=32.45, lon=-93.84


In [4]:
import numpy as np
from metpy.calc import thickness_hydrostatic
from metpy.units import units
from metpy.constants import earth_gravity, dry_air_gas_constant

# this cell tries to more closely recreate https://github.com/ana-v-espinoza/tropopauseCalc/issues/4
# version of the function. 
# this version of the function runs a lot slower than the one above. 

def wmo(pFull, TFull, lapseC=2.0*units("K/km"), height=False):
    """
    Implements NCAR's Fortran code in python:
        https://github.com/NCAR/ncl/blob/develop/ni/src/lib/nfpfort/stattrop_dp.f
    """

    nLev = pFull.size
    nLevm = nLev-1

    pMin = 85.0*units.mbar
    pMax = 450.0*units.mbar

    dZ = 2000.0*units.meters

    g = earth_gravity
    R = dry_air_gas_constant

    const = g/R

    found = False

    lapse = np.zeros(nLevm)*units.kelvin/units.km
    pHalf = np.zeros(nLevm)*units.mbar
    pTrop = 0*units.mbar
    for iLev in range(0, nLevm):
        lapse[iLev] = const*np.log(TFull[iLev]/TFull[iLev+1])/np.log(pFull[iLev]/pFull[iLev+1])
        pHalf[iLev] = (pFull[iLev]+pFull[iLev+1])*0.5

    for iLev in range(0, nLevm - 1):
        if lapse[iLev] < lapseC and pFull[iLev] < pMax and not found:
            P1 = np.log(pHalf[iLev].magnitude)
            P2 = np.log(pHalf[iLev + 1].magnitude)
            if lapse[iLev] != lapse[iLev + 1]:
                weight = (lapseC - lapse[iLev]) / (lapse[iLev + 1] - lapse[iLev])
                pTrop = np.exp(P1 + weight * (P2 - P1)) * units.mbar
            else:
                pTrop = pHalf[iLev]
    
            p2km = pTrop * np.exp(-dZ * const / TFull[iLev])
            lapseSum = 0
            kount = 0
            for L in range(iLev, nLevm):
                if pHalf[L] > p2km:
                    lapseSum += lapse[L]
                    kount += 1
            lapseAvg = lapseSum / kount if kount > 0 else lapse[iLev]
            found = lapseAvg < lapseC
            if found:
                iTrop = iLev
                pTrop = pMin if pTrop < pMin else pTrop
                break
    
    # Only print if not found after the loop
    if not found:
        print("Tropopause not found")

    if height:
        z = thickness_hydrostatic(pFull[0:iTrop],TFull[0:iTrop])
        return z.to(units.km)

    return pTrop.to(units.mbar)

df = pd.read_csv("/file_path")
p = df["pressure_hPa"].values * units.hPa
T = (df["temperature_C"].values + 273.15) * units.kelvin

ptrop = wmo(p, T, lapseC=2.0 * units("K/km"), height=False)
print(f"Tropopause pressure: {ptrop:.2f}")

/home/lthompson/miniconda3/envs/melodies-monet-develop3/lib/python3.11/site-packages/pint/facets/plain/quantity.py:1006: RuntimeWarning: divide by zero encountered in scalar divide
  magnitude = magnitude_op(new_self._magnitude, other._magnitude)
/home/lthompson/miniconda3/envs/melodies-monet-develop3/lib/python3.11/site-packages/pint/facets/plain/quantity.py:1006: RuntimeWarning: invalid value encountered in scalar divide
  magnitude = magnitude_op(new_self._magnitude, other._magnitude)


Tropopause pressure: nan millibar
